# IPI-persistence — single-model run on Kaggle (Qwen2.5-3B-Instruct)

Runs the whole pipeline (Phase A -> states -> Phase B -> clean -> judge -> **analysis**) on **one
open-weight model**, locally on the Kaggle GPU. **No API keys, no cost.** Aligned with repo commit
that introduced context-aware markers and the `clean_control_conditions` config key.

## HOW TO RUN OVERNIGHT WITHOUT LOSING RESULTS  (read this first)

Running interactively (the blue "Run All") and closing your laptop **will lose everything**: Kaggle
deletes `/kaggle/working` when an interactive session ends. Instead:

1. Right sidebar: **Accelerator = GPU T4 x1**, **Internet = On**.
2. Top-right: **Save Version -> "Save & Run All (Commit)" -> Save**.
3. This runs **headless on Kaggle's servers** — you can close the tab / laptop / lose wifi and it
   keeps going, then **saves all output permanently**. Watch it under the **"Versions"** menu.
4. When the version finishes, open it -> **Output** tab -> download `runs_checkpoint.zip`
   (contains `results/`, the per-call `cache/`, and the full `analysis/` report + tables + figures).

**Commit time limit (important):** a commit runs for at most ~12 h; if it *overruns* it is marked
failed and its output is **not** saved. So pick a profile that finishes inside 12 h (see Cell 4):
`"quick"` reliably finishes in one overnight commit; `"full"` is the preregistered design and needs
**several commits** — after each committed version, open a **new version**, **Add Input -> the previous
version's output**, and commit again. Cell 1 restores `runs/` and finished calls are served from the
cache, so each commit resumes where the last one stopped instead of redoing work.

A `runs_checkpoint.zip` is rewritten after every step, and the pipeline writes append-only JSONL, so
the data on disk is always current — the resume above never repeats finished model calls.

In [ ]:
# --- Cell 1: get the repo + restore any previous checkpoint (for resume across sessions) ---
import os, sys, subprocess, glob, shutil

REPO_URL  = "https://github.com/kritimathurug23-cpu/ipi-persistence.git"
REPO_DIR  = "/kaggle/working/ipi-persistence"
WORK_RUNS = "/kaggle/working/runs"

if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
    if r.returncode != 0:
        # Fallback: repo added as a Kaggle dataset (use if the GitHub repo is private)
        cands = glob.glob("/kaggle/input/*/ipi-persistence") + glob.glob("/kaggle/input/*/*/ipi-persistence")
        assert cands, "Clone failed and no repo dataset found. Enable Internet, or add the repo as a dataset."
        shutil.copytree(cands[0], REPO_DIR)
        print("Copied repo from", cands[0])
print("Repo:", REPO_DIR, "->", sorted(os.listdir(REPO_DIR))[:10])

# Restore a checkpoint from a previously-saved version output attached as an input dataset.
# (Add Input -> your previous version's Output; it contains the whole /kaggle/working, incl. runs/.)
if not os.path.isdir(WORK_RUNS):
    restored = False
    for cand in glob.glob("/kaggle/input/*/runs") + glob.glob("/kaggle/input/*/*/runs"):
        shutil.copytree(cand, WORK_RUNS); print("Restored checkpoint from", cand); restored = True; break
    if not restored:
        print("No prior checkpoint found - starting fresh.")
else:
    print("Existing checkpoint in", WORK_RUNS)

In [ ]:
# --- Cell 2: dependencies (torch is preinstalled on Kaggle GPU) ---
!pip -q install pyyaml pandas numpy scipy statsmodels matplotlib lifelines langdetect scikit-learn tabulate
!pip -q install -U "transformers>=4.44" accelerate
print("dependencies ready")

In [ ]:
# --- Cell 3: load the model and route provider 'local_hf' to it (no repo edits, no API keys) ---
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"   # ungated; same model as before. Swap for any HF instruct model.
print("Loading", MODEL_ID, "...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto").eval()
except TypeError:  # older transformers
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto").eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

@torch.inference_mode()
def _local_hf(model_id, system, messages, settings):
    msgs = ([{"role": "system", "content": system}] if system else []) + list(messages)
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    max_new = int(settings.get("max_tokens", settings.get("max_new_tokens", 512)))
    gen = dict(max_new_tokens=max_new, pad_token_id=tokenizer.pad_token_id)
    # Default to the model developer's recommended sampling (its generation_config), as preregistered.
    # An explicit setting overrides it; temperature 0 = greedy.
    if "temperature" in settings:
        temp = float(settings["temperature"])
        if temp > 0:
            gen.update(do_sample=True, temperature=temp, top_p=float(settings.get("top_p", 1.0)))
        else:
            gen.update(do_sample=False)
    out = model.generate(**enc, **gen)
    gen_ids = out[0][enc["input_ids"].shape[1]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
    return {"text": text, "usage": {"input": int(enc["input_ids"].shape[1]), "output": int(gen_ids.shape[0])}}

sys.path.insert(0, REPO_DIR)
import harness.providers as P
_orig_chat = P.chat
def _chat(provider, model_id, system, messages, settings, base_url=None, api_key_env=None, **kwargs):
    # **kwargs: the harness may pass options such as prompt_caching; accept and forward them.
    if provider == "local_hf":
        return _local_hf(model_id, system, messages, settings)
    return _orig_chat(provider, model_id, system, messages, settings, base_url, api_key_env, **kwargs)
P.chat = _chat                                            # harness.calls calls providers.chat dynamically
print("Default sampling settings:", {k: getattr(model.generation_config, k, None)
      for k in ("do_sample", "temperature", "top_p", "top_k", "repetition_penalty")})
print("Model ready; provider 'local_hf' patched in.")

In [ ]:
# --- Cell 4: choose a profile, write the Kaggle config, smoke-test one generation ---
import yaml

# PROFILE picks the amount of work:
#   "quick" = light 1-rep / 4-turn pass over all scenarios. RELIABLY finishes in ONE overnight
#             commit (well under the ~12 h limit). Recommended for a first, safe, complete result.
#   "full"  = preregistered pilot design (6 turns, 2 reps, clean controls, ~40 states/model);
#             matches config/pilot_open.yaml. Best data, but ~18-20 h -> needs SEVERAL commits with
#             the resume flow (a single commit that overruns 12 h fails and saves nothing).
PROFILE = "quick"
PROFILES = {
    "full":  dict(n_subsequent_turns=6, reps_phase_a=2, reps_phase_b=2, clean_reps=3,
                  clean_control_conditions=["P3"], target_states_per_model=40),
    "quick": dict(n_subsequent_turns=4, reps_phase_a=1, reps_phase_b=1, clean_reps=1,
                  clean_control_conditions=[],    target_states_per_model=6),
}
P4 = PROFILES[PROFILE]

CFG = {
    "run_dir": WORK_RUNS + "/pilot_qwen3b",      # absolute -> lands in persistent /kaggle/working
    "scenarios_dir": "data/scenarios",
    "unrelated_tasks": "data/unrelated_tasks.json",
    "system_prompt": "You are a helpful assistant.",
    "n_subsequent_turns": P4["n_subsequent_turns"],
    "reps_phase_a": P4["reps_phase_a"], "reps_phase_b": P4["reps_phase_b"], "clean_reps": P4["clean_reps"],
    "clean_control_conditions": P4["clean_control_conditions"],   # new repo key (CLEAN_P3 fair baseline)
    "target_states_per_model": P4["target_states_per_model"], "constructed_max_attempts": 3,
    "cross_task_conditions": ["P2", "P3"], "default_task_type": "same_topic",
    "models": [{"name": "qwen3b", "provider": "local_hf", "model_id": MODEL_ID,
                "settings": {"max_tokens": 512}}],
    # No judge: the judge must NOT be one of the evaluated models, and there is no API model here.
    # The harness skips the judge step when judge_model is null.
    "judge_model": None,
}
CFG_PATH = os.path.join(REPO_DIR, "config", "kaggle_qwen3b.yaml")
with open(CFG_PATH, "w") as f:
    yaml.safe_dump(CFG, f, sort_keys=False)
print(f"Profile: {PROFILE}  ->  {P4}")
print("Wrote", CFG_PATH)
print("Smoke test:", _local_hf(MODEL_ID, "You are a helpful assistant.",
      [{"role": "user", "content": "Say hello in one short sentence."}], {"max_tokens": 32})["text"])

In [ ]:
# --- Cell 5: run the pipeline (resumable). Each step checkpoints; re-running skips finished work. ---
os.chdir(REPO_DIR)
sys.stderr = sys.stdout          # harness prints progress to stderr; merge so lines appear in order
LIMIT = None                     # all 60 attack + 12 benign scenarios. Set e.g. LIMIT = 5 to wire-test.

from harness.run import Ctx, phase_a, build_states, phase_b, clean, judge
ctx = Ctx(CFG_PATH, models_filter=None, limit=LIMIT)

for name, fn in [("phase-a", phase_a), ("states", build_states),
                 ("phase-b", phase_b), ("clean", clean), ("judge", judge)]:
    print(f"\n===== {name} =====", flush=True)
    fn(ctx)                                                    # cached calls + existing rows are skipped
    shutil.make_archive("/kaggle/working/runs_checkpoint", "zip", WORK_RUNS)   # checkpoint after each step
print("\nPipeline done. Checkpoint: /kaggle/working/runs_checkpoint.zip")

In [ ]:
# --- Cell 6: quick results + FULL analysis (reads results only; no model needed), then re-checkpoint ---
import pandas as pd
from harness.calls import read_jsonl

RES = CFG["run_dir"] + "/results"
pb = pd.DataFrame(read_jsonl(RES + "/phase_b.jsonl"))
print("phase_b rows:", len(pb))
if len(pb):
    d = pb[pb.task_type == "same_topic"]
    print("\nAttacker-marker rate by condition (mean over turns, %):")
    print((d.assign(m=d.marker.astype(float)).groupby("condition").m.mean() * 100).round(1).to_string())
    ck = d.dropna(subset=["constraint_kept"]).copy()
    if len(ck):
        ck["ck"] = ck.constraint_kept.astype(float)
        print("\nConstraint-kept / RQ2 rate by condition (%):")
        print((ck.groupby("condition").ck.mean() * 100).round(1).to_string())

# Full evaluation: tables + figures + report.md written under runs/.../analysis/
try:
    subprocess.run([sys.executable, "-m", "analysis.analyze", "--config", CFG_PATH], cwd=REPO_DIR, check=True)
    print("\nAnalysis written to", CFG["run_dir"] + "/analysis")
except Exception as e:
    print("\nanalysis.analyze may be limited on a tiny test set:", repr(e))

# Re-checkpoint so the zip now also contains the analysis/ output (report, tables, figures).
shutil.make_archive("/kaggle/working/runs_checkpoint", "zip", WORK_RUNS)
print("Final checkpoint (incl. analysis): /kaggle/working/runs_checkpoint.zip")

In [ ]:
# --- Cell 7: one-click download (interactive sessions) ---
# In a COMMITTED version you download from the version's Output tab instead; this link is for when
# you are watching an interactive session and want the zip immediately.
import os
from IPython.display import FileLink, display
z = "/kaggle/working/runs_checkpoint.zip"
if os.path.exists(z):
    print("size (MB):", round(os.path.getsize(z) / 1e6, 2))
    display(FileLink(z))          # click to download runs_checkpoint.zip
else:
    print("No checkpoint yet - run Cell 5 first.")

## Notes

- **Model:** `Qwen/Qwen2.5-3B-Instruct` (ungated, fits a T4). Change `MODEL_ID` in Cell 3 for any HF
  instruct model; larger ones may need 4-bit (`bitsandbytes`) or a bigger accelerator.
- **Profiles (Cell 4):** `"quick"` (default) reliably finishes in one overnight commit and is a
  complete pass over all 72 scenarios at light settings; `"full"` is the preregistered pilot design
  (best data) but must be run over several commits with the resume flow (a commit over ~12 h fails).
- **Scoring is now context-aware:** a marker phrase that is quoted, negated, corrected, or attributed
  to the document does **not** count as attack success (repo `scoring/markers.py`); the raw rule is kept
  as `marker_raw` for comparison. Re-running this notebook picks that up automatically (it clones the repo).
- **No judge:** the judge must not be one of the evaluated models and there is no API model here, so the
  judge step is skipped (`judge_model: null`).
- **Checkpoint / resume:** results + the per-call cache + the analysis live in `/kaggle/working/runs`.
  Save Version so it persists; to resume, attach that version's Output as an input dataset and commit
  again — Cell 1 restores it and finished calls come from the cache.
- **Merging with the other model later (optional):** unzip this and the Llama-8B checkpoint locally, then
  `python -m scripts.merge_runs runs/pilot_llama8b runs/pilot_qwen3b --into runs/pilot` and
  `python -m analysis.analyze --config config/pilot_open.yaml`.